In [1]:
%matplotlib inline
import math
import random
import re
import warnings

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

## 2 序列模型

### 2.1 理论计算题

<img src="./hw04_pic/2-1.png" width="400">

### 2.2 编程题

In [2]:
import re
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除标点符号，只保留字母和空格
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    # 2. 按空格分词
    tokens = text.split()

    # 3. 按出现频率排序构建词汇表，分配整数 ID
    word_counts = Counter(tokens)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, count) in enumerate(sorted_words)}

    # 4. 滑动窗口生成特征序列和对应标签
    features = []
    labels = []

    for i in range(len(tokens) - n + 1):
        feature = tokens[i:i+n]
        label = tokens[i+n] if i+n < len(tokens) else None

        features.append(feature)
        labels.append(label)

    return vocab, (features, labels)


vocab, (features, labels) = preprocess_text("The time machine", n=2)

print("案例1：The time machine")
print("词表:", vocab)
print("特征:", features)
print("标签:", labels)

print("-" * 40)

vocab, (features, labels) = preprocess_text("I like deep learning very much", n=2)

print("案例2：I like deep learning very much")
print("词表:", vocab)
print("特征:", features)
print("标签:", labels)

案例1：The time machine
词表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]
----------------------------------------
案例2：I like deep learning very much
词表: {'deep': 0, 'i': 1, 'learning': 2, 'like': 3, 'much': 4, 'very': 5}
特征: [['i', 'like'], ['like', 'deep'], ['deep', 'learning'], ['learning', 'very'], ['very', 'much']]
标签: ['deep', 'learning', 'very', 'much', None]


## 3 循环神经网络

### 3.1 理论计算题

<img src="./hw04_pic/3-1.png" width="600">

### 3.2 编程题

In [3]:
import torch

class ManualRNNCell:
    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size

        # 初始化输入到隐藏层、隐藏层到隐藏层的权重和偏置
        self.W_hx = torch.randn(input_size, hidden_size) * 0.1
        self.W_hh = torch.randn(hidden_size, hidden_size) * 0.1
        self.b_h = torch.zeros(hidden_size)

        # 保存前向传播中的中间变量，供反向传播使用
        self.cache = None

    def forward(self, x_t, h_prev):
        # 计算当前时刻的线性变换结果
        a_t = x_t @ self.W_hx + h_prev @ self.W_hh + self.b_h

        # 使用 tanh 激活函数得到当前隐藏状态
        h_t = torch.tanh(a_t)

        # 保存反向传播需要用到的变量
        self.cache = (x_t, h_prev, h_t)

        return h_t

    def backward(self, dh_next):
        # 取出前向传播保存的变量
        x_t, h_prev, h_t = self.cache

        # tanh 的导数为 1 - h_t^2
        da = dh_next * (1 - h_t ** 2)

        # 计算输入、上一隐藏状态、权重和偏置的梯度
        dx_t = da @ self.W_hx.T
        dh_prev = da @ self.W_hh.T
        dW_hx = x_t.T @ da
        dW_hh = h_prev.T @ da
        db_h = da.sum(dim=0)

        return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 构造测试数据
batch_size, input_size, hidden_size = 2, 3, 4

cell = ManualRNNCell(input_size, hidden_size)

x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)
dh_next = torch.randn(batch_size, hidden_size)

# 前向传播
h_t = cell.forward(x_t, h_prev)

# 单步反向传播
dx_t, dh_prev, dW_hx, dW_hh, db_h = cell.backward(dh_next)

# 打印各结果的形状
print("h_t 形状:", tuple(h_t.shape))
print("dx_t 形状:", tuple(dx_t.shape))
print("dh_prev 形状:", tuple(dh_prev.shape))
print("dW_hx 形状:", tuple(dW_hx.shape))
print("dW_hh 形状:", tuple(dW_hh.shape))
print("db_h 形状:", tuple(db_h.shape))

h_t 形状: (2, 4)
dx_t 形状: (2, 3)
dh_prev 形状: (2, 4)
dW_hx 形状: (3, 4)
dW_hh 形状: (4, 4)
db_h 形状: (4,)


## 4 高级循环神经网络

### 4.1 理论计算题

<img src="./hw04_pic/4-1.png" width="600">

### 4.2 编程题

In [4]:
import torch
import torch.nn as nn

# 构造输入数据
seq_len, batch_size, input_dim, hidden_dim = 5, 3, 4, 6
X = torch.randn(seq_len, batch_size, input_dim)

# 定义单层双向 RNN
birnn = nn.RNN(
    input_size=input_dim,
    hidden_size=hidden_dim,
    num_layers=1,
    bidirectional=True,
)

# Y 是每个时间步的双向隐藏状态拼接结果
# h_n 是最终隐藏状态
Y, h_n = birnn(X)

# 从 Y 中拆出每个时间步的前向隐藏状态和后向隐藏状态
forward_outputs = Y[:, :, :hidden_dim]
backward_outputs = Y[:, :, hidden_dim:]

# h_n[0] 是前向 RNN 的最终隐藏状态
# h_n[1] 是后向 RNN 的最终隐藏状态
forward_final = h_n[0]
backward_final = h_n[1]

# 拼接前向和后向最终隐藏状态，作为整个序列表示
sequence_rep = torch.cat((forward_final, backward_final), dim=1)

print("输入形状:", tuple(X.shape))

print("前向所有时间步隐藏状态形状:", tuple(forward_outputs.shape))
print("后向所有时间步隐藏状态形状:", tuple(backward_outputs.shape))
print("拼接后所有时间步隐藏状态 Y 形状:", tuple(Y.shape))

print("前向最终隐藏状态形状:", tuple(forward_final.shape))
print("后向最终隐藏状态形状:", tuple(backward_final.shape))
print("拼接后的序列表示形状:", tuple(sequence_rep.shape))

print("前向所有时间步隐藏状态:")
print(forward_outputs)

print("后向所有时间步隐藏状态:")
print(backward_outputs)

print("拼接后所有时间步隐藏状态 Y:")
print(Y)

print("拼接后的序列表示 sequence_rep:")
print(sequence_rep)

输入形状: (5, 3, 4)
前向所有时间步隐藏状态形状: (5, 3, 6)
后向所有时间步隐藏状态形状: (5, 3, 6)
拼接后所有时间步隐藏状态 Y 形状: (5, 3, 12)
前向最终隐藏状态形状: (3, 6)
后向最终隐藏状态形状: (3, 6)
拼接后的序列表示形状: (3, 12)
前向所有时间步隐藏状态:
tensor([[[ 0.3207,  0.3015, -0.6509, -0.5982, -0.4495,  0.6729],
         [-0.1250,  0.7874, -0.1370, -0.2342, -0.9250,  0.3028],
         [-0.6482,  0.1216, -0.6202, -0.8241, -0.0967,  0.2620]],

        [[ 0.2628, -0.3295, -0.8013, -0.6505, -0.4052,  0.6887],
         [ 0.6666, -0.0315, -0.4763, -0.3115, -0.0332,  0.7772],
         [-0.1415, -0.2953, -0.6959, -0.6369, -0.4570,  0.6547]],

        [[ 0.3366, -0.1536, -0.6775, -0.4225, -0.7404,  0.5471],
         [-0.3724,  0.0735, -0.6935, -0.6940, -0.7617,  0.2629],
         [ 0.0363,  0.3997, -0.3958, -0.3469, -0.9202,  0.2913]],

        [[ 0.5415,  0.0104, -0.3249, -0.1704, -0.7146,  0.3471],
         [ 0.3327,  0.2198, -0.4722, -0.3214, -0.7591,  0.5801],
         [-0.4594,  0.1601, -0.7301, -0.7982, -0.2021,  0.5114]],

        [[ 0.2192,  0.4556, -0.0844,  0.0426,

## 5 嵌入向量

### 5.1 理论计算题

<img src="./hw04_pic/5-1.png" width="600">

### 5.2 编程题

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        # 输入词向量矩阵 W，形状为 (V, d)
        self.W = nn.Parameter(torch.randn(vocab_size, embed_dim) * 0.01)

        # 输出权重矩阵 W_out，形状为 (d, V)
        self.W_out = nn.Parameter(torch.randn(embed_dim, vocab_size) * 0.01)

    def forward(self, contexts):
        # 根据上下文词索引查找词向量
        embeds = self.W[contexts]              # (batch, context_size, d)

        # 对上下文词向量取平均，得到隐藏层表示
        hidden = embeds.mean(dim=1)            # (batch, d)

        # 计算输出层得分
        logits = hidden @ self.W_out           # (batch, V)

        # 完整 softmax，得到每个词作为中心词的概率
        probs = F.softmax(logits, dim=1)       # (batch, V)

        return hidden, logits, probs

    def loss(self, contexts, targets):
        hidden, logits, probs = self.forward(contexts)

        # 交叉熵损失，目标为真实中心词索引
        ce_loss = F.cross_entropy(logits, targets)

        return ce_loss, hidden, logits, probs


# 构造测试数据
vocab_size, embed_dim = 8, 5
contexts = torch.tensor([[1, 2, 3, 4], [0, 2, 5, 6]])
targets = torch.tensor([3, 1])

cbow = CBOW(vocab_size, embed_dim)

loss, hidden, logits, probs = cbow.loss(contexts, targets)

print("contexts 形状:", tuple(contexts.shape))
print("targets 形状:", tuple(targets.shape))
print("hidden 形状:", tuple(hidden.shape))
print("logits 形状:", tuple(logits.shape))
print("probs 形状:", tuple(probs.shape))
print("loss =", float(loss))

print("hidden:")
print(hidden)

print("logits:")
print(logits)

print("probs:")
print(probs)

contexts 形状: (2, 4)
targets 形状: (2,)
hidden 形状: (2, 5)
logits 形状: (2, 8)
probs 形状: (2, 8)
loss = 2.079446315765381
hidden:
tensor([[ 0.0015, -0.0064, -0.0069,  0.0031, -0.0051],
        [ 0.0020,  0.0070, -0.0025, -0.0038,  0.0010]],
       grad_fn=<MeanBackward1>)
logits:
tensor([[ 4.6231e-05, -7.4898e-05, -2.0177e-04,  1.5121e-05, -8.8625e-05,
         -1.3382e-05,  4.9780e-05, -3.3023e-05],
        [ 7.0596e-05, -3.2026e-05, -9.6309e-06, -4.7301e-05,  6.6098e-05,
          2.6611e-05,  1.0996e-04,  5.8569e-05]], grad_fn=<MmBackward0>)
probs:
tensor([[0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]],
       grad_fn=<SoftmaxBackward0>)


## 6 注意力机制

### 6.1 理论计算题

<img src="./hw04_pic/6-1.png" width="400">

### 6.2 编程题

In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 分别对输入进行线性变换，得到 Q、K、V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # 多头拼接后的输出线性层
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, X):
        # X: (seq_len, batch, d_model)
        seq_len, batch, d_model = X.shape

        # 拆成多个注意力头
        X = X.reshape(seq_len, batch, self.num_heads, self.d_k)

        # 调整维度为 (batch, num_heads, seq_len, d_k)
        return X.permute(1, 2, 0, 3)

    def combine_heads(self, X):
        # X: (batch, num_heads, seq_len, d_k)
        batch, num_heads, seq_len, d_k = X.shape

        # 调整回 (seq_len, batch, num_heads, d_k)
        X = X.permute(2, 0, 1, 3).contiguous()

        # 拼接所有头，得到 (seq_len, batch, d_model)
        return X.reshape(seq_len, batch, num_heads * d_k)

    def forward(self, X):
        # 线性投影并分头
        Q = self.split_heads(self.W_q(X))
        K = self.split_heads(self.W_k(X))
        V = self.split_heads(self.W_v(X))

        # 计算缩放点积注意力得分
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 对最后一维做 softmax，得到注意力权重
        attn_weights = F.softmax(scores, dim=-1)

        # 根据注意力权重对 V 加权求和
        context = torch.matmul(attn_weights, V)

        # 拼接多个头
        out = self.combine_heads(context)

        # 经过最终线性层，输出形状与输入相同
        out = self.W_o(out)

        return out, attn_weights


# 构造测试数据
seq_len, batch, d_model = 6, 2, 4
X = torch.randn(seq_len, batch, d_model)

mha = MultiHeadAttention(d_model=4, num_heads=2)

Y, attn = mha(X)

print("输入形状:", tuple(X.shape))
print("输出形状:", tuple(Y.shape))
print("注意力权重形状:", tuple(attn.shape))

print("输出 Y:")
print(Y)

print("注意力权重 attn:")
print(attn)

输入形状: (6, 2, 4)
输出形状: (6, 2, 4)
注意力权重形状: (2, 2, 6, 6)
输出 Y:
tensor([[[-0.4244, -0.0885,  0.0775,  0.2377],
         [-0.4520, -0.1163,  0.0982,  0.3001]],

        [[-0.4072, -0.0829,  0.0557,  0.2182],
         [-0.3883, -0.1291,  0.0333,  0.2983]],

        [[-0.3138, -0.0916,  0.0036,  0.1881],
         [-0.3332, -0.1340, -0.0131,  0.2850]],

        [[-0.4466, -0.0878,  0.0964,  0.2474],
         [-0.3407, -0.1296,  0.0044,  0.2796]],

        [[-0.4398, -0.0816,  0.0793,  0.2316],
         [-0.4301, -0.1221,  0.0590,  0.3027]],

        [[-0.3589, -0.1001,  0.0531,  0.2277],
         [-0.4371, -0.1226,  0.0668,  0.3062]]], grad_fn=<ViewBackward0>)
注意力权重 attn:
tensor([[[[0.1741, 0.2335, 0.0998, 0.2192, 0.1826, 0.0908],
          [0.1760, 0.2014, 0.1194, 0.1990, 0.1916, 0.1126],
          [0.1558, 0.1367, 0.2034, 0.1403, 0.1512, 0.2125],
          [0.1722, 0.2581, 0.0776, 0.2375, 0.1867, 0.0679],
          [0.1779, 0.2259, 0.0861, 0.2221, 0.2106, 0.0774],
          [0.1657, 0.1964, 